In [ ]:
# Installing necessary packages to run future code

%pip install pandas numpy scikit-learn matplotlib
%pip install openpyxl
%pip install scikit-learn

In [ ]:
# Organising environmental sensor data so that it is in a nice format

import json
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")  # put your three files in a "data" folder next to this script

# 1. Load files (default C engine; no rows are silently skipped)
environmental_df = pd.read_csv(DATA_DIR / "Environmental sensor.csv")

# Ensure the columns index and row index are flat and clean
environmental_df.columns = [str(col) for col in environmental_df.columns]
environmental_df.reset_index(drop=True, inplace=True)

# 2. Define the JSON parsing helper function
def parse_sensor_json(json_str):
    if pd.isna(json_str):
        return {}
    try:
        if isinstance(json_str, str):
            if json_str.startswith('""') or '""' in json_str:
                json_str = json_str.replace('""', '"')
            if json_str.startswith('"') and json_str.endswith('"'):
                json_str = json_str[1:-1]
            data = json.loads(json_str)
        else:
            data = json_str
            
        extracted = {}
        if isinstance(data, list):
            for item in data:
                var_info = item.get('variable', {})
                var_name = var_info.get('name')
                values_list = item.get('values', [])
                if values_list and isinstance(values_list, list):
                    metric_value = values_list[0].get('value')
                    if var_name is not None:
                        extracted[var_name] = metric_value
        return extracted
    except Exception:
        return {}

# 3. Parse and extract key-value metrics from environmental data
parsed_metrics = environmental_df['jsondata'].apply(parse_sensor_json)
metrics_df = pd.DataFrame(parsed_metrics.tolist())
metrics_df.reset_index(drop=True, inplace=True)

# Map standard headings to include units explicitly
unit_mapping = {
    'Battery': 'Battery (V)',
    'Carbon dioxide': 'Carbon dioxide (ppm)',
    'Humidity': 'Humidity (%)',
    'Formaldehyde': 'Formaldehyde (mg/m³)',
    'Pressure': 'Pressure (mb)',
    'Temperature': 'Temperature (°C)',
    'Light': 'Light (lx)',
    'Light Volatile Organic Compounds': 'Light Volatile Organic Compounds (ppb)',
    'Particulate matter 1': 'Particulate matter 1 (µg/m³)',
    'Particulate matter 4': 'Particulate matter 4 (µg/m³)',
    'Particulate matter 2.5': 'Particulate matter 2.5 (µg/m³)',
    'Particulate matter 10': 'Particulate matter 10 (µg/m³)'
}
metrics_df.rename(columns=unit_mapping, inplace=True)

# 4. Combine extracted metrics with the original DataFrame (dropping original raw JSON column)
environmental_expanded_df = pd.concat(
    [environmental_df.drop(columns=['jsondata']), metrics_df],
    axis=1
)

# 5. Display summaries and verify columns directly
print("--- Verified Column Names in Environmental Data ---")
for col in environmental_expanded_df.columns:
    print(f" - {col}")

print("\n--- Expanded Environmental Sensor Data with Units ---")
display(environmental_expanded_df.head())



In [ ]:
# Shuffle all rows into a random order and split 75% / 25%
# random_state makes the shuffle reproducible (same split every run)

data_lot_1 = environmental_expanded_df.sample(frac=0.75, random_state=42)           # 75% split
data_lot_2 = environmental_expanded_df.drop(data_lot_1.index).sample(frac=1, random_state=42)  # remaining 25%, also shuffled

# NOTE: random_state=42 keeps the data split the same for multiple reruns of the data

print(f"Total rows:  {len(environmental_expanded_df)}")
print(f"Data lot 1:  {len(data_lot_1)} ({len(data_lot_1)/len(environmental_expanded_df):.1%})")
print(f"Data lot 2:  {len(data_lot_2)} ({len(data_lot_2)/len(environmental_expanded_df):.1%})")
print("Any overlap:", bool(set(data_lot_1.index) & set(data_lot_2.index)))



In [ ]:
# Multiple linear regression code

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Find a column by its name, ignoring the unit in brackets (avoids unit-label mismatches)
def find_col(df, name):
    matches = [c for c in df.columns if c.startswith(name + " (")]
    if len(matches) != 1:
        raise KeyError(f"Expected one column for '{name}', found {matches}")
    return matches[0]

predictor_names = ["Carbon dioxide", "Temperature", "Humidity", "Pressure"]
target_names = ["Formaldehyde", "Light Volatile Organic Compounds",
                "Particulate matter 1", "Particulate matter 2.5",
                "Particulate matter 4", "Particulate matter 10"]

X_cols = [find_col(data_lot_1, n) for n in predictor_names]
y_cols = [find_col(data_lot_1, n) for n in target_names]

models = {}     # fitted models, kept so they can be applied to data lot 2 later
bounds = {}     # (lower, upper) physical limits per target, taken from the training data
results = []
coefs = {}

for y_col in y_cols:
    # Only rows where this target was actually measured can be used for training
    train = data_lot_1[X_cols + [y_col]].dropna()
    X_train, y_train = train[X_cols], train[y_col]

    model = LinearRegression().fit(X_train, y_train)

    # Physical constraint: concentrations cannot be negative, and estimates
    # should not exceed the largest value actually observed in training
    lower = max(0.0, y_train.min())
    upper = y_train.max()
    bounds[y_col] = (lower, upper)

    pred_raw = model.predict(X_train)
    pred = np.clip(pred_raw, lower, upper)

    models[y_col] = model
    coefs[y_col] = pd.Series(model.coef_, index=X_cols)
    results.append({
        "target": y_col,
        "rows_used": len(train),
        "intercept": model.intercept_,
        "R2_raw": r2_score(y_train, pred_raw),
        "R2_clipped": r2_score(y_train, pred),
        "MAE_raw": mean_absolute_error(y_train, pred_raw),
        "MAE_clipped": mean_absolute_error(y_train, pred),
        "RMSE_clipped": np.sqrt(mean_squared_error(y_train, pred)),
        "mean_baseline_MAE": mean_absolute_error(y_train, np.full(len(y_train), y_train.mean())),
        "pct_rows_clipped": 100 * np.mean((pred_raw < lower) | (pred_raw > upper)),
    })

print(pd.DataFrame(results).round(3).to_string(index=False))
print(pd.DataFrame(coefs).round(4))


def predict_rows(df):
    """Estimate every target for every row of df, clipped to physical limits.

    Parameters
    ----------
    df : DataFrame containing the four predictor columns (X_cols).

    Returns
    -------
    DataFrame with the id (if present), the four predictors, and one
    'Estimated <target>' column per target. Rows with a missing predictor
    get NaN estimates.
    """
    out = df[(["id"] if "id" in df.columns else []) + X_cols].copy()
    valid = df[X_cols].notna().all(axis=1)
    for y_col in y_cols:
        lower, upper = bounds[y_col]
        est = pd.Series(np.nan, index=df.index)
        est[valid] = np.clip(models[y_col].predict(df.loc[valid, X_cols]), lower, upper)
        out["Estimated " + y_col] = est
    return out


# Estimates for every row in data lot 1
estimates_lot_1 = predict_rows(data_lot_1)
print(estimates_lot_1.head().round(2).to_string())

In [ ]:
# Linear interpolation code

